In [1]:
import os, glob
import pandas as pd
import librosa
import cv2
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoProcessor, AutoImageProcessor

# from datasets import Dataset
from torch.utils.data import DataLoader , Dataset


tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
audio_processor = AutoProcessor.from_pretrained("facebook/hubert-large-ls960-ft")
video_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")

class MultiModalDataset(Dataset):
    def __init__(self, data_root, split, modal="all", tokenizer=None, audio_processor=None, video_processor=None):
        assert split in ["train", "val", "test"]
        assert modal in ["audio", "video", "text", "all"]
        self.modal = modal

        df = pd.read_csv(os.path.join(data_root, f"{split}.csv") , sep = "\t")
        sid = df["sentence_id"].values
        text = df["text"].values
        label = df["label"].values
        assert len(sid) == len(label)

        if self.modal in ["text", "all"]:
            assert tokenizer is not None
            self.tokenizer = tokenizer

        if self.modal in ["audio", "all"]:
            assert audio_processor is not None
            audio_root = os.path.join(data_root, "audio")
            self.audio_processor = audio_processor
            self.sampling_rate = self.audio_processor.feature_extractor.sampling_rate

        if self.modal in ["video", "all"]:
            assert video_processor is not None
            video_root = os.path.join(data_root, "video")
            self.video_processor = video_processor
            self.n_imgs = 16

        self.data_list = []
        for (_sid, _text, _label) in tqdm(zip(sid, text, label), total=len(sid)):
            data = { "label": _label }
            video_id = "_".join(_sid.split("_")[:-2])
            if self.modal in ["text", "all"]:
                text_ids = self.tokenizer(_text)
                data["text"] = text_ids
            if self.modal in ["audio", "all"]:
                audio_path = os.path.join(audio_root, video_id, f"{_sid}.mp3")
                data["audio"] = audio_path
            if self.modal in ["video", "all"]:
                img_paths = []
                for i in range(self.n_imgs):
                    img_path = os.path.join(video_root, video_id, _sid, f"{i}.jpg")
                    if not os.path.exists(img_path): break
                    img_paths.append(img_path)
                if len(img_paths) < self.n_imgs: continue
                data["video"] = img_paths
            self.data_list.append(data)


    def load_audio(self, path):
        audio, _ = librosa.load(path, sr=self.sampling_rate)
        X = self.audio_processor(audio, sampling_rate=self.sampling_rate, return_tensors="pt").input_values
        return X
    
    def load_images(self, paths):
        imgs = [cv2.imread(path)[:,:,::-1] for path in paths]
        X = self.video_processor(imgs, return_tensors="pt")
        return X

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        data = self.data_list[idx]
        out_dict = { "label": data["label"] }
        if "text" in data:
            out_dict["text"] = data["text"]
        if "audio" in data:
            out_dict["audio"] = self.load_audio(data["audio"])
        if "video" in data:
            out_dict["video"] = self.load_images(data["video"])
        return out_dict
 
def collate_fn(batch):
    out_tuple = ()
    if "text" in batch[0]:
        input_ids = [torch.tensor(x["text"]["input_ids"], dtype=torch.long) for x in batch ]
        attention_mask = [ torch.tensor(x["text"]["attention_mask"], dtype=torch.long) for x in batch]
        text = {
            "input_ids": torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0),
            "attention_mask": torch.nn.utils.rnn.pad_sequence( attention_mask, batch_first=True, padding_value=0),
        }
        out_tuple += (text,)
    if "audio" in batch[0]:
        audio_seqs = [ torch.tensor(x["audio"][0], dtype=torch.float) for x in batch]
        audio = torch.nn.utils.rnn.pad_sequence( audio_seqs, batch_first=True, padding_value=0.0)
        out_tuple += (audio,)
    if "video" in batch[0]:
        video = {
            "pixel_values": torch.stack([x["video"]["pixel_values"][0] for x in batch]),
        }
        out_tuple += (video,)
    label = torch.tensor([x["label"] for x in batch])
    out_tuple += (label,)
    return out_tuple


if __name__ == "__main__":
    data_root = "./dataset"

    test_set = MultiModalDataset(data_root, "test", modal="all", tokenizer=tokenizer, audio_processor=audio_processor, video_processor=video_processor)

    test_loader = DataLoader(test_set, batch_size=2, shuffle=False, collate_fn=collate_fn)

    text, audio, video, label = next(iter(test_loader))

    print(text["input_ids"].shape)
    print(audio.shape)
    print(video["pixel_values"].shape)
    print(label)

/home/mudasir/miniconda3/envs/mm-turn-taking/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
100%|██████████| 9186/9186 [00:01<00:00, 6236.63it/s]


torch.Size([2, 38])
torch.Size([2, 188433])
torch.Size([2, 16, 3, 224, 224])
tensor([0, 0])


/tmp/ipykernel_1396603/1199777904.py:103: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio_seqs = [ torch.tensor(x["audio"][0], dtype=torch.float) for x in batch]


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.nn.parameter import Parameter
from torch.nn.init import xavier_normal_, kaiming_normal_
import math


class LMF(nn.Module):
    """
    Low-rank Multimodal Fusion
    """
    def __init__(self, hidden_dim=256, output_dim=3, rank=16, use_softmax=False, post_fusion_prob=0.1):
        super(LMF, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.rank = rank
        self.use_softmax = use_softmax
        self.post_fusion_prob = post_fusion_prob
        
        self.post_fusion_dropout = nn.Dropout(p=self.post_fusion_prob)
        self.factor_1 = Parameter(torch.Tensor(self.rank, self.hidden_dim + 1, self.output_dim))
        self.factor_2 = Parameter(torch.Tensor(self.rank, self.hidden_dim + 1, self.output_dim))
        self.factor_3 = Parameter(torch.Tensor(self.rank, self.hidden_dim + 1, self.output_dim))
        self.fusion_weights = Parameter(torch.Tensor(1, self.rank))
        self.fusion_bias = Parameter(torch.Tensor(1, self.output_dim))
        
        xavier_normal_(self.factor_1)
        xavier_normal_(self.factor_2)
        xavier_normal_(self.factor_3)
        xavier_normal_(self.fusion_weights)
        self.fusion_bias.data.fill_(0)
    
    def forward(self, text_x, audio_x, video_x):
        temp_x = text_x if text_x is not None else audio_x
        batch_size = temp_x.data.shape[0]
        DTYPE = torch.cuda.FloatTensor if temp_x.is_cuda else torch.FloatTensor
        
        if text_x is not None:
            _text_h = torch.cat((Variable(torch.ones(batch_size, 1).type(DTYPE), requires_grad=False), text_x), dim=1)
            fusion_text = torch.matmul(_text_h, self.factor_1)
        
        if audio_x is not None:
            _audio_h = torch.cat((Variable(torch.ones(batch_size, 1).type(DTYPE), requires_grad=False), audio_x), dim=1)
            fusion_audio = torch.matmul(_audio_h, self.factor_2)
        
        if video_x is not None:
            _video_h = torch.cat((Variable(torch.ones(batch_size, 1).type(DTYPE), requires_grad=False), video_x), dim=1)
            fusion_video = torch.matmul(_video_h, self.factor_3)
        
        if text_x is None:
            fusion_zy = fusion_audio * fusion_video
        elif audio_x is None:
            fusion_zy = fusion_text * fusion_video
        elif video_x is None:
            fusion_zy = fusion_audio * fusion_text
        else:
            fusion_zy = fusion_audio * fusion_video * fusion_text
        
        output = torch.matmul(self.fusion_weights, fusion_zy.permute(1, 0, 2)).squeeze() + self.fusion_bias
        output = output.view(-1, self.output_dim)
        
        if self.use_softmax:
            output = F.softmax(output, dim=-1)
        
        return output


class EarlyFusion(nn.Module):
    """
    Early Fusion: Simple concatenation followed by MLP
    """
    def __init__(self, hidden_dim=256, output_dim=3, dropout=0.1):
        super(EarlyFusion, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # MLP for fusion
        self.fc1 = nn.Linear(hidden_dim * 3, hidden_dim * 2)
        self.fc2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
    def forward(self, text_x, audio_x, video_x):
        # Handle missing modalities by using zeros
        batch_size = (text_x if text_x is not None else 
                     audio_x if audio_x is not None else video_x).shape[0]
        device = (text_x if text_x is not None else 
                 audio_x if audio_x is not None else video_x).device
        
        if text_x is None:
            text_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if audio_x is None:
            audio_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if video_x is None:
            video_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        
        # Concatenate all modalities
        fused = torch.cat([text_x, audio_x, video_x], dim=1)
        
        # MLP
        x = self.relu(self.fc1(fused))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.layer_norm(x)
        x = self.dropout(x)
        output = self.fc3(x)
        
        return output


class LateFusion(nn.Module):
    """
    Late Fusion: Independent processing followed by weighted combination
    """
    def __init__(self, hidden_dim=256, output_dim=3, dropout=0.1):
        super(LateFusion, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Individual classifiers for each modality
        self.text_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        
        self.audio_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        
        self.video_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        
        # Learnable weights for each modality
        self.weights = nn.Parameter(torch.ones(3))
        
    def forward(self, text_x, audio_x, video_x):
        outputs = []
        active_weights = []
        
        if text_x is not None:
            outputs.append(self.text_classifier(text_x))
            active_weights.append(self.weights[0])
        
        if audio_x is not None:
            outputs.append(self.audio_classifier(audio_x))
            active_weights.append(self.weights[1])
        
        if video_x is not None:
            outputs.append(self.video_classifier(video_x))
            active_weights.append(self.weights[2])
        
        # Weighted average
        active_weights = torch.stack(active_weights)
        active_weights = F.softmax(active_weights, dim=0)
        
        output = sum(w * o for w, o in zip(active_weights, outputs))
        
        return output


class TensorFusionNetwork(nn.Module):
    """
    Tensor Fusion Network: Outer product of all modalities
    """
    def __init__(self, hidden_dim=256, output_dim=3, dropout=0.1):
        super(TensorFusionNetwork, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Post-fusion dimensions
        post_fusion_dim = (hidden_dim + 1) ** 3
        
        self.post_fusion_dropout = nn.Dropout(p=dropout)
        self.post_fusion_layer_1 = nn.Linear(post_fusion_dim, hidden_dim)
        self.post_fusion_layer_2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.post_fusion_layer_3 = nn.Linear(hidden_dim // 2, output_dim)
        
    def forward(self, text_x, audio_x, video_x):
        batch_size = (text_x if text_x is not None else 
                     audio_x if audio_x is not None else video_x).shape[0]
        device = (text_x if text_x is not None else 
                 audio_x if audio_x is not None else video_x).device
        
        # Add constant 1 dimension for bias
        if text_x is None:
            text_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if audio_x is None:
            audio_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if video_x is None:
            video_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        
        # Add the constant 1
        text_x = torch.cat([torch.ones(batch_size, 1).to(device), text_x], dim=1)
        audio_x = torch.cat([torch.ones(batch_size, 1).to(device), audio_x], dim=1)
        video_x = torch.cat([torch.ones(batch_size, 1).to(device), video_x], dim=1)
        
        # Compute outer product
        fusion_tensor = torch.bmm(text_x.unsqueeze(2), audio_x.unsqueeze(1))
        fusion_tensor = fusion_tensor.view(batch_size, -1, 1)
        fusion_tensor = torch.bmm(fusion_tensor, video_x.unsqueeze(1))
        fusion_tensor = fusion_tensor.view(batch_size, -1)
        
        # Post-fusion layers
        x = self.post_fusion_dropout(fusion_tensor)
        x = F.relu(self.post_fusion_layer_1(x))
        x = self.post_fusion_dropout(x)
        x = F.relu(self.post_fusion_layer_2(x))
        output = self.post_fusion_layer_3(x)
        
        return output


class MultimodalFactorizedBilinear(nn.Module):
    """
    Multimodal Factorized Bilinear Pooling (MFB)
    """
    def __init__(self, hidden_dim=256, output_dim=3, mfb_factor=5, dropout=0.1):
        super(MultimodalFactorizedBilinear, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.mfb_factor = mfb_factor
        self.mfb_out_dim = hidden_dim
        
        # Text-Audio MFB
        self.text_audio_proj1 = nn.Linear(hidden_dim, self.mfb_out_dim * mfb_factor)
        self.text_audio_proj2 = nn.Linear(hidden_dim, self.mfb_out_dim * mfb_factor)
        
        # Audio-Video MFB
        self.audio_video_proj1 = nn.Linear(hidden_dim, self.mfb_out_dim * mfb_factor)
        self.audio_video_proj2 = nn.Linear(hidden_dim, self.mfb_out_dim * mfb_factor)
        
        # Text-Video MFB
        self.text_video_proj1 = nn.Linear(hidden_dim, self.mfb_out_dim * mfb_factor)
        self.text_video_proj2 = nn.Linear(hidden_dim, self.mfb_out_dim * mfb_factor)
        
        # Final fusion
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.mfb_out_dim * 3, output_dim)
        
    def mfb_pooling(self, x1, x2, proj1, proj2):
        """Perform MFB pooling between two modalities"""
        z1 = proj1(x1)
        z2 = proj2(x2)
        
        z1 = z1.view(-1, self.mfb_factor, self.mfb_out_dim)
        z2 = z2.view(-1, self.mfb_factor, self.mfb_out_dim)
        
        # Element-wise product and sum over factor dimension
        z = (z1 * z2).sum(1)
        z = torch.sqrt(F.relu(z)) - torch.sqrt(F.relu(-z))
        z = F.normalize(z, p=2, dim=1)
        
        return z
    
    def forward(self, text_x, audio_x, video_x):
        batch_size = (text_x if text_x is not None else 
                     audio_x if audio_x is not None else video_x).shape[0]
        device = (text_x if text_x is not None else 
                 audio_x if audio_x is not None else video_x).device
        
        if text_x is None:
            text_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if audio_x is None:
            audio_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if video_x is None:
            video_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        
        # Pairwise MFB pooling
        ta_fused = self.mfb_pooling(text_x, audio_x, self.text_audio_proj1, self.text_audio_proj2)
        av_fused = self.mfb_pooling(audio_x, video_x, self.audio_video_proj1, self.audio_video_proj2)
        tv_fused = self.mfb_pooling(text_x, video_x, self.text_video_proj1, self.text_video_proj2)
        
        # Concatenate all fused features
        fused = torch.cat([ta_fused, av_fused, tv_fused], dim=1)
        fused = self.dropout(fused)
        
        output = self.fc(fused)
        
        return output


class CrossModalAttention(nn.Module):
    """
    Cross-Modal Attention Fusion
    """
    def __init__(self, hidden_dim=256, output_dim=3, num_heads=4, dropout=0.1):
        super(CrossModalAttention, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        
        # Multi-head attention for cross-modal interactions
        self.text_audio_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.text_video_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.audio_video_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        
        # Final fusion layers
        self.fusion_fc = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, text_x, audio_x, video_x):
        batch_size = (text_x if text_x is not None else 
                     audio_x if audio_x is not None else video_x).shape[0]
        device = (text_x if text_x is not None else 
                 audio_x if audio_x is not None else video_x).device
        
        if text_x is None:
            text_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if audio_x is None:
            audio_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if video_x is None:
            video_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        
        # Reshape for attention (add sequence dimension)
        text_x = text_x.unsqueeze(1)
        audio_x = audio_x.unsqueeze(1)
        video_x = video_x.unsqueeze(1)
        
        # Cross-modal attention
        ta_out, _ = self.text_audio_attn(text_x, audio_x, audio_x)
        ta_out = self.norm1(ta_out + text_x)
        
        tv_out, _ = self.text_video_attn(text_x, video_x, video_x)
        tv_out = self.norm2(tv_out + text_x)
        
        av_out, _ = self.audio_video_attn(audio_x, video_x, video_x)
        av_out = self.norm3(av_out + audio_x)
        
        # Concatenate and squeeze
        fused = torch.cat([ta_out, tv_out, av_out], dim=-1).squeeze(1)
        
        output = self.fusion_fc(fused)
        
        return output


class GatedMultimodalUnit(nn.Module):
    """
    Gated Multimodal Unit (GMU)
    """
    def __init__(self, hidden_dim=256, output_dim=3, dropout=0.1):
        super(GatedMultimodalUnit, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Gating mechanisms for each modality
        self.text_gate = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.Sigmoid()
        )
        
        self.audio_gate = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.Sigmoid()
        )
        
        self.video_gate = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.Sigmoid()
        )
        
        # Transform layers
        self.text_transform = nn.Linear(hidden_dim, hidden_dim)
        self.audio_transform = nn.Linear(hidden_dim, hidden_dim)
        self.video_transform = nn.Linear(hidden_dim, hidden_dim)
        
        # Final classifier
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        
    def forward(self, text_x, audio_x, video_x):
        batch_size = (text_x if text_x is not None else 
                     audio_x if audio_x is not None else video_x).shape[0]
        device = (text_x if text_x is not None else 
                 audio_x if audio_x is not None else video_x).device
        
        if text_x is None:
            text_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if audio_x is None:
            audio_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if video_x is None:
            video_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        
        # Concatenate all modalities for gating
        concat_features = torch.cat([text_x, audio_x, video_x], dim=1)
        
        # Compute gates
        text_gate = self.text_gate(concat_features)
        audio_gate = self.audio_gate(concat_features)
        video_gate = self.video_gate(concat_features)
        
        # Apply gates to transformed features
        text_h = text_gate * torch.tanh(self.text_transform(text_x))
        audio_h = audio_gate * torch.tanh(self.audio_transform(audio_x))
        video_h = video_gate * torch.tanh(self.video_transform(video_x))
        
        # Fuse gated features
        fused = text_h + audio_h + video_h
        fused = self.dropout(fused)
        
        output = self.fc(fused)
        
        return output


class MultimodalTransformer(nn.Module):
    """
    Multimodal Transformer with self-attention across modalities
    """
    def __init__(self, hidden_dim=256, output_dim=3, num_heads=4, num_layers=2, dropout=0.1):
        super(MultimodalTransformer, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Modality embeddings
        self.text_embed = nn.Linear(hidden_dim, hidden_dim)
        self.audio_embed = nn.Linear(hidden_dim, hidden_dim)
        self.video_embed = nn.Linear(hidden_dim, hidden_dim)
        
        # Positional/modality type embeddings
        self.modality_embedding = nn.Embedding(3, hidden_dim)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output projection
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, text_x, audio_x, video_x):
        batch_size = (text_x if text_x is not None else 
                     audio_x if audio_x is not None else video_x).shape[0]
        device = (text_x if text_x is not None else 
                 audio_x if audio_x is not None else video_x).device
        
        modalities = []
        modality_ids = []
        
        if text_x is not None:
            text_emb = self.text_embed(text_x).unsqueeze(1)
            modalities.append(text_emb)
            modality_ids.append(0)
        
        if audio_x is not None:
            audio_emb = self.audio_embed(audio_x).unsqueeze(1)
            modalities.append(audio_emb)
            modality_ids.append(1)
        
        if video_x is not None:
            video_emb = self.video_embed(video_x).unsqueeze(1)
            modalities.append(video_emb)
            modality_ids.append(2)
        
        # Concatenate modalities
        x = torch.cat(modalities, dim=1)
        
        # Add modality type embeddings
        modality_ids = torch.tensor(modality_ids).to(device)
        modality_embs = self.modality_embedding(modality_ids).unsqueeze(0).expand(batch_size, -1, -1)
        x = x + modality_embs
        
        # Transformer encoding
        x = self.transformer(x)
        
        # Global average pooling
        x = x.mean(dim=1)
        x = self.dropout(x)
        
        output = self.fc(x)
        
        return output


class TuckerFusion(nn.Module):
    """
    Tucker Decomposition Fusion (Generalization of LMF)
    """
    def __init__(self, hidden_dim=256, output_dim=3, rank=(16, 16, 16), dropout=0.1):
        super(TuckerFusion, self).__init__()
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.rank = rank
        
        # Tucker decomposition factors
        self.text_factor = Parameter(torch.Tensor(rank[0], hidden_dim + 1))
        self.audio_factor = Parameter(torch.Tensor(rank[1], hidden_dim + 1))
        self.video_factor = Parameter(torch.Tensor(rank[2], hidden_dim + 1))
        
        # Core tensor
        self.core_tensor = Parameter(torch.Tensor(rank[0], rank[1], rank[2]))
        
        # Post-fusion layers
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(rank[0] * rank[1] * rank[2], hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
        
        # Initialize
        xavier_normal_(self.text_factor)
        xavier_normal_(self.audio_factor)
        xavier_normal_(self.video_factor)
        xavier_normal_(self.core_tensor)
        
    def forward(self, text_x, audio_x, video_x):
        batch_size = (text_x if text_x is not None else 
                     audio_x if audio_x is not None else video_x).shape[0]
        device = (text_x if text_x is not None else 
                 audio_x if audio_x is not None else video_x).device
        
        if text_x is None:
            text_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if audio_x is None:
            audio_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        if video_x is None:
            video_x = torch.zeros(batch_size, self.hidden_dim).to(device)
        
        # Add bias term
        text_x = torch.cat([torch.ones(batch_size, 1).to(device), text_x], dim=1)
        audio_x = torch.cat([torch.ones(batch_size, 1).to(device), audio_x], dim=1)
        video_x = torch.cat([torch.ones(batch_size, 1).to(device), video_x], dim=1)
        
        # Project to low-rank space
        text_proj = torch.matmul(text_x, self.text_factor.t())  # (batch, rank[0])
        audio_proj = torch.matmul(audio_x, self.audio_factor.t())  # (batch, rank[1])
        video_proj = torch.matmul(video_x, self.video_factor.t())  # (batch, rank[2])
        
        # Tucker contraction with core tensor
        fusion = torch.einsum('bi,ijk,bj,bk->b', 
                             text_proj, self.core_tensor, audio_proj, video_proj)
        
        # For better stability, we can also use sequential contraction
        # Reshape for batch processing
        fusion = torch.einsum('bi,bj,bk->bijk', text_proj, audio_proj, video_proj)
        fusion = fusion.view(batch_size, -1)
        
        fusion = self.dropout(fusion)
        output = self.fc(fusion)
        
        return output

def get_fusion_module(module_name):
    mechanisms = {
        "LMF": LMF(),
        "Early Fusion": EarlyFusion(),
        "Late Fusion": LateFusion(),
        "TFN": TensorFusionNetwork(),
        "MFB": MultimodalFactorizedBilinear(),
        "Cross-Modal Attention": CrossModalAttention(),
        "GMU": GatedMultimodalUnit(),
        "Multimodal Transformer": MultimodalTransformer(),
        "Tucker Fusion": TuckerFusion()
    }
    return mechanisms[module_name] 


# Testing code
if __name__ == "__main__":
    batch_size = 16
    hidden_dim = 256
    output_dim = 3
    
    # Create dummy data
    text = torch.randn(batch_size, hidden_dim)
    audio = torch.randn(batch_size, hidden_dim)
    video = torch.randn(batch_size, hidden_dim)
    
    print("Testing all fusion mechanisms:\n")
    
    # Test each fusion mechanism
    mechanisms = {
        "LMF": LMF(),
        "Early Fusion": EarlyFusion(),
        "Late Fusion": LateFusion(),
        "TFN": TensorFusionNetwork(),
        "MFB": MultimodalFactorizedBilinear(),
        "Cross-Modal Attention": CrossModalAttention(),
        "GMU": GatedMultimodalUnit(),
        "Multimodal Transformer": MultimodalTransformer(),
        "Tucker Fusion": TuckerFusion()
    }
    
    for name, model in mechanisms.items():
        print(f"{name}:")
        print(f"  All modalities: {model(text, audio, video).shape}")
        print(f"  Missing text:   {model(None, audio, video).shape}")
        print(f"  Missing audio:  {model(text, None, video).shape}")
        print(f"  Missing video:  {model(text, audio, None).shape}")
        
        # Count parameters
        params = sum(p.numel() for p in model.parameters())
        print(f"  Parameters: {params:,}")
        print()

Testing all fusion mechanisms:

LMF:
  All modalities: torch.Size([16, 3])
  Missing text:   torch.Size([16, 3])
  Missing audio:  torch.Size([16, 3])
  Missing video:  torch.Size([16, 3])
  Parameters: 37,027

Early Fusion:
  All modalities: torch.Size([16, 3])
  Missing text:   torch.Size([16, 3])
  Missing audio:  torch.Size([16, 3])
  Missing video:  torch.Size([16, 3])
  Parameters: 526,339

Late Fusion:
  All modalities: torch.Size([16, 3])
  Missing text:   torch.Size([16, 3])
  Missing audio:  torch.Size([16, 3])
  Missing video:  torch.Size([16, 3])
  Parameters: 99,852

TFN:
  All modalities: torch.Size([16, 3])
  Missing text:   torch.Size([16, 3])
  Missing audio:  torch.Size([16, 3])
  Missing video:  torch.Size([16, 3])
  Parameters: 4,345,529,347

MFB:
  All modalities: torch.Size([16, 3])
  Missing text:   torch.Size([16, 3])
  Missing audio:  torch.Size([16, 3])
  Missing video:  torch.Size([16, 3])
  Parameters: 1,976,067

Cross-Modal Attention:
  All modalities: torc

In [28]:
import torch
from torch import nn

from transformers import AutoTokenizer, GPT2Model
from transformers import AutoProcessor, HubertModel
from transformers import AutoImageProcessor, VideoMAEModel

from model.fusion import LMF
from model.fusion2 import get_fusion_module


class LanguageModel(nn.Module):
    def __init__(self, pretrained_model_name_or_path="openai-community/gpt2", return_embeddings=False):
        super(LanguageModel, self).__init__()
        self.transformer = GPT2Model.from_pretrained(pretrained_model_name_or_path)
        self.return_embeddings = return_embeddings
        hidden_size = self.transformer.config.n_embd
        self.proj = nn.Linear(hidden_size, 256)
        self.out_layer = nn.Linear(256, 3)
        # for i, layer in enumerate(self.transformer.transformer.h):
        #     if i < len(self.transformer.transformer.h) - 2:
        #         for param in layer.parameters():
        #             param.requires_grad = False
        # for param in self.transformer.wte.parameters():
        #     param.requires_grad = False
        # for param in self.transformer.wpe.parameters():
        #     param.requires_grad = False
        # for param in self.transformer.ln_f.parameters():
        #     param.requires_grad = False

    def forward(self, inputs):
        hidden_state = self.transformer(inputs).last_hidden_state #(bs, len, 768)
        last_hidden_state = hidden_state[:, -1, :]
        proj = self.proj(last_hidden_state)
        if self.return_embeddings: return proj
        out = self.out_layer(proj)
        return out
    

class AudioModel(nn.Module):
    def __init__(self, pretrained_model_name_or_path="facebook/hubert-base-ls960", return_embeddings=False):
        super(AudioModel, self).__init__()
        self.hubert = HubertModel.from_pretrained(pretrained_model_name_or_path)
        self.return_embeddings = return_embeddings
        hidden_size = self.hubert.config.hidden_size
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Linear(hidden_size, 256)
        self.out_layer = nn.Linear(256, 3)

        # for i, layer in enumerate(self.hubert.encoder.layers):
        #     if i < len(self.hubert.encoder.layers) - 2:
        #         for param in layer.parameters():
        #             param.requires_grad = False

        # for param in self.hubert.feature_extractor.parameters():
        #     param.requires_grad = False
        # for param in self.hubert.feature_projection.parameters():
        #     param.requires_grad = False

    def forward(self, inputs):
        hidden_state = self.hubert(inputs).last_hidden_state
        avg_pooled = self.avg_pool(hidden_state.transpose(1, 2)).squeeze(-1)
        proj = self.proj(avg_pooled)
        if self.return_embeddings: return proj
        out = self.out_layer(proj)
        return out
    

class VisionModel(nn.Module):
    def __init__(self, pretrained_model_name_or_path="MCG-NJU/videomae-base", return_embeddings=False):
        super(VisionModel, self).__init__()
        self.model = VideoMAEModel.from_pretrained(pretrained_model_name_or_path)
        self.return_embeddings = return_embeddings
        hidden_size = self.model.config.hidden_size
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Linear(hidden_size, 256)
        self.out_layer = nn.Linear(256, 3)

        # for i, layer in enumerate(self.model.encoder.layer):
        #     if i < len(self.model.encoder.layer) - 2:
        #         for param in layer.parameters():
        #             param.requires_grad = False
        # for param in self.model.embeddings.parameters():
        #     param.requires_grad = False
        # for param in self.model.layernorm.parameters():
        #     param.requires_grad = False

    def forward(self, inputs):
        hidden_state = self.model(inputs).last_hidden_state
        avg_pooled = self.avg_pool(hidden_state.transpose(1, 2)).squeeze(-1)
        proj = self.proj(avg_pooled)
        if self.return_embeddings: return proj
        out = self.out_layer(proj)
        return out
    

class LanguageAudioVisionModel(nn.Module):
    def __init__(self, text_ckpt_path=None, audio_ckpt_path=None, vision_ckpt_path=None , fusion_module = None):
        super(LanguageAudioVisionModel, self).__init__()

        self.text_model = LanguageModel(return_embeddings=True)
        if text_ckpt_path is not None:
            self.text_model.load_state_dict(torch.load(text_ckpt_path), strict=False)
        self.audio_model = AudioModel(return_embeddings=True)
        if audio_ckpt_path is not None:
            self.audio_model.load_state_dict(torch.load(audio_ckpt_path), strict=False)
        self.vision_model = VisionModel(return_embeddings=True)
        if vision_ckpt_path is not None:
            self.vision_model.load_state_dict(torch.load(vision_ckpt_path), strict=False)

        self.fusion_module = fusion_module
        if self.fusion_module:
            self.fusion = get_fusion_module(fusion_module)
        else:
            self.fusion = LMF()
    def forward(self, text_inputs, audio_inputs, vision_inputs):
        text_proj = self.text_model(text_inputs) if text_inputs is not None else None
        audio_proj = self.audio_model(audio_inputs) if audio_inputs is not None else None
        vision_proj = self.vision_model(vision_inputs) if vision_inputs is not None else None

        out = self.fusion(text_proj, audio_proj, vision_proj)
        
        return out
    

def load_mm_model(args):
    model = LanguageAudioVisionModel(args.fusion_module).to(args.device)
    model.load_state_dict(torch.load(args.ckpt_path), strict=False)
    model.eval()
    return model


def text_processor(text):
    text = text.strip().lower()
    symbols = [".", ",", "!", "?", ":", ";", "(", ")", "[", "]", "{", "}", "<", ">", "\"", "'"]
    for symbol in symbols:
        text = text.replace(symbol, "")
    return text


def load_processors():
    tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    audio_processor = AutoProcessor.from_pretrained("facebook/hubert-large-ls960-ft")
    video_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
    return tokenizer, text_processor, audio_processor, video_processor

In [13]:
args = {
    "data_root" : "./dataset/",
    "modal" : "audio",
    "batch_size" : 1,
    "n_workers" : 0,
    "lr" : 1e-5,
    "device" : "cuda",
    "n_epoch" : 10,
    "log_dir" : "./log",
    "random_drop_modal_rate" : 0.05
}

In [14]:
tokenizer, _, audio_processor, video_processor = load_processors()
train_set = MultiModalDataset(
    data_root=args["data_root"],
    split="train",
    modal=args["modal"],
    tokenizer=tokenizer,
    audio_processor=audio_processor,
    video_processor=video_processor,
)
val_set = MultiModalDataset(
    data_root=args["data_root"],
    split="val",
    modal=args["modal"],
    tokenizer=tokenizer,
    audio_processor=audio_processor,
    video_processor=video_processor,
)

In [15]:
train_loader = DataLoader(train_set, batch_size=args["batch_size"], shuffle=True, collate_fn=collate_fn, num_workers=args["n_workers"])
val_loader = DataLoader(val_set, batch_size=args["batch_size"], shuffle=False, collate_fn=collate_fn, num_workers=args["n_workers"])

In [16]:
model = AudioModel(return_embeddings=False).to("cuda")

In [17]:
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
criterion = nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import time
import os
from tqdm import tqdm 
t_bar = tqdm(range(args['n_epoch']))
for epoch in t_bar:
    # train
    model.train()
    n_batch = len(train_loader)
    epoch_loss = 0.0
    all_labels, all_logits = [], []
    for i, (X, y) in tqdm(enumerate(train_loader)):
        if X is None:
            continue
        if args["modal"] == "text":
            X = X["input_ids"].to("cuda")
        elif args["modal"] == "audio":
            X = X.to("cuda")
        elif args["modal"] == "video":
            X = X["pixel_values"].to("cuda")
        y = y.to("cuda")
        optimizer.zero_grad()
        
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        all_labels.extend(y.detach().cpu().numpy())
        all_logits.extend(pred.argmax(dim=1).detach().cpu().numpy())

        t_bar.set_description(f"Epoch {epoch} training | Batch {i}/{n_batch} | Loss {loss.item():.4f}")
        epoch_loss += (loss.item() * args["batch_size"])
    epoch_loss = epoch_loss / n_batch
    
    all_labels , all_logits = np.array(all_labels), np.array(all_logits)
    accuracy = accuracy_score(all_labels, all_logits)
    recall = recall_score(all_labels, all_logits, average=None)
    f1 = f1_score(all_labels, all_logits, average=None)
    precision = precision_score(all_labels, all_logits, average=None)
    print(f"accuracy : {accuracy} , recall : {recall} , precision : {precision} , f1 : {f1}")
    
    # val
    model.eval()
    n_batch = len(val_loader)
    val_loss = 0.0
    all_labels, all_logits = [], []
    for i, (X, y) in enumerate(train_loader):
        if X is None:
            continue
        if args["modal"] == "text":
            X = X["input_ids"].to("cuda")
        elif args["modal"] == "audio":
            X = X.to("cuda")
        elif args["modal"] == "video":
            X = X["pixel_values"].to("cuda")
        y = y.to("cuda")
        with torch.no_grad():
            pred = model(X)
            loss = criterion(pred, y)
            
            all_labels.extend(y.detach().cpu().numpy())
            all_logits.extend(pred.argmax(dim=1).detach().cpu().numpy())
            val_loss += (loss.item() * args["batch_size"])
    val_loss = val_loss / n_batch
    
    all_labels , all_logits = np.array(all_labels), np.array(all_logits)
    accuracy = accuracy_score(all_labels, all_logits)
    recall = recall_score(all_labels, all_logits, average=None)
    f1 = f1_score(all_labels, all_logits, average=None)
    precision = precision_score(all_labels, all_logits, average=None)
    print(f"accuracy : {accuracy} , recall : {recall} , precision : {precision} , f1 : {f1}")
    
    time_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()).replace(" ", "_")
    run_dir = f"./runs/{time_str}_{args['modal']}"
    os.makedirs(run_dir, exist_ok=True)
    save_path = os.path.join(run_dir, f"epoch_{epoch}.pt")
    torch.save(model.state_dict(), save_path)
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**2:.2f} MB")


In [20]:
from model.mm import LanguageAudioVisionModel, load_processors
model = LanguageAudioVisionModel()

In [21]:
model

LanguageAudioVisionModel(
  (text_model): LanguageModel(
    (transformer): GPT2Model(
      (wte): Embedding(50257, 768)
      (wpe): Embedding(1024, 768)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=2304, nx=768)
            (c_proj): Conv1D(nf=768, nx=768)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=3072, nx=768)
            (c_proj): Conv1D(nf=768, nx=3072)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (proj): Linear(in_features=768, 

In [22]:
multimodal_ckpt_path = "./multi-modal-deid.pt"
output_dir = "./model_weights"
checkpoint = torch.load(multimodal_ckpt_path, map_location='cpu')
os.makedirs(output_dir, exist_ok=True)


In [23]:
# Extract text model weights
text_state_dict = {}
for key, value in checkpoint.items():
    if key.startswith('text_model.'):
        # Remove 'text_model.' prefix
        new_key = key.replace('text_model.', '')
        text_state_dict[new_key] = value

# Extract audio model weights
audio_state_dict = {}
for key, value in checkpoint.items():
    if key.startswith('audio_model.'):
        # Remove 'audio_model.' prefix
        new_key = key.replace('audio_model.', '')
        audio_state_dict[new_key] = value

# Extract vision model weights
vision_state_dict = {}
for key, value in checkpoint.items():
    if key.startswith('vision_model.'):
        # Remove 'vision_model.' prefix
        new_key = key.replace('vision_model.', '')
        vision_state_dict[new_key] = value

In [24]:
# Save extracted weights
text_path = os.path.join(output_dir, 'text_model.pt')
audio_path = os.path.join(output_dir, 'audio_model.pt')
vision_path = os.path.join(output_dir, 'vision_model.pt')

In [25]:
torch.save(text_state_dict, text_path)
print(f"✓ Saved text model weights to {text_path}")
print(f"  - Found {len(text_state_dict)} parameters")

torch.save(audio_state_dict, audio_path)
print(f"✓ Saved audio model weights to {audio_path}")
print(f"  - Found {len(audio_state_dict)} parameters")

torch.save(vision_state_dict, vision_path)
print(f"✓ Saved vision model weights to {vision_path}")
print(f"  - Found {len(vision_state_dict)} parameters")

✓ Saved text model weights to ./model_weights/text_model.pt
  - Found 150 parameters
✓ Saved audio model weights to ./model_weights/audio_model.pt
  - Found 213 parameters
✓ Saved vision model weights to ./model_weights/vision_model.pt
  - Found 186 parameters


In [26]:
# Print sample keys for verification
print("\nSample text model keys:")
for i, key in enumerate(list(text_state_dict.keys())[:5]):
    print(f"  {key}")

print("\nSample audio model keys:")
for i, key in enumerate(list(audio_state_dict.keys())[:5]):
    print(f"  {key}")

print("\nSample vision model keys:")
for i, key in enumerate(list(vision_state_dict.keys())[:5]):
    print(f"  {key}")


Sample text model keys:
  transformer.wte.weight
  transformer.wpe.weight
  transformer.h.0.ln_1.weight
  transformer.h.0.ln_1.bias
  transformer.h.0.attn.c_attn.weight

Sample audio model keys:
  hubert.masked_spec_embed
  hubert.feature_extractor.conv_layers.0.conv.weight
  hubert.feature_extractor.conv_layers.0.layer_norm.weight
  hubert.feature_extractor.conv_layers.0.layer_norm.bias
  hubert.feature_extractor.conv_layers.1.conv.weight

Sample vision model keys:
  model.embeddings.patch_embeddings.projection.weight
  model.embeddings.patch_embeddings.projection.bias
  model.encoder.layer.0.attention.attention.q_bias
  model.encoder.layer.0.attention.attention.v_bias
  model.encoder.layer.0.attention.attention.query.weight


In [29]:
from model.mm import LanguageModel, AudioModel, VisionModel

In [30]:
print("\n1. Testing text model...")
text_model = LanguageModel(return_embeddings=False)
text_state = torch.load(text_path, map_location='cpu')
result = text_model.load_state_dict(text_state, strict=False)
print(f"   ✓ Text model loaded successfully")
if result.missing_keys:
    print(f"   ⚠ Missing keys: {result.missing_keys}")
if result.unexpected_keys:
    print(f"   ⚠ Unexpected keys: {result.unexpected_keys}")


1. Testing text model...
   ✓ Text model loaded successfully
   ⚠ Missing keys: ['out_layer.weight', 'out_layer.bias']


In [31]:
print("\n2. Testing audio model...")
audio_model = AudioModel(return_embeddings=False)
audio_state = torch.load(audio_path, map_location='cpu')
result = audio_model.load_state_dict(audio_state, strict=False)
print(f"   ✓ Audio model loaded successfully")
if result.missing_keys:
    print(f"   ⚠ Missing keys: {result.missing_keys}")
if result.unexpected_keys:
    print(f"   ⚠ Unexpected keys: {result.unexpected_keys}")


2. Testing audio model...
   ✓ Audio model loaded successfully
   ⚠ Missing keys: ['out_layer.weight', 'out_layer.bias']


In [32]:
print("\n3. Testing vision model...")
vision_model = VisionModel(return_embeddings=False)
vision_state = torch.load(vision_path, map_location='cpu')
result = vision_model.load_state_dict(vision_state, strict=False)
print(f"   ✓ Vision model loaded successfully")
if result.missing_keys:
    print(f"   ⚠ Missing keys: {result.missing_keys}")
if result.unexpected_keys:
    print(f"   ⚠ Unexpected keys: {result.unexpected_keys}")

print("\n✓ All models verified successfully!")


3. Testing vision model...
   ✓ Vision model loaded successfully
   ⚠ Missing keys: ['out_layer.weight', 'out_layer.bias']

✓ All models verified successfully!


In [33]:
tokenizer, _, audio_processor, video_processor = load_processors()
train_set = MultiModalDataset(
    data_root=args.data_root,
    split="train",
    tokenizer=tokenizer,
    audio_processor=audio_processor,
    video_processor=video_processor,
)
val_set = MultiModalDataset(
    data_root=args.data_root,
    split="val",
    tokenizer=tokenizer,
    audio_processor=audio_processor,
    video_processor=video_processor,
)

train_loader = DataLoader(train_set, batch_size=args["batch_size"], shuffle=True, collate_fn=collate_fn, num_workers=args["n_workers"])
val_loader = DataLoader(val_set, batch_size=args["batch_size"], shuffle=False, collate_fn=collate_fn, num_workers=args["n_workers"])

model = LanguageAudioVisionModel(args.fusion_module).to(args.device)
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
criterion = nn.CrossEntropyLoss()

AttributeError: 'dict' object has no attribute 'data_root'

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import time
import os
import random


t_bar = tqdm(range(args['n_epoch']))
for epoch in t_bar:
    # train
    model.train()
    n_batch = len(train_loader)
    epoch_loss = 0.0
    all_labels, all_logits = [], []
    for i, (text, audio, vision, y) in enumerate(train_loader):
        if audio is None:
            continue
        text = text["input_ids"].to("cuda")
        audio = audio.to("cuda")
        vision = vision["pixel_values"].to("cuda")
        y = y.to("cuda")
        optimizer.zero_grad()
        
        if random.random() < random_drop_modal_rate:
            drop_modal = random.choice(["text", "audio", "vision"])
            if drop_modal == "text":
                text = None
            elif drop_modal == "audio":
                audio = None
            elif drop_modal == "vision":
                vision = None
            # print("drop modal:", drop_modal)
        pred = model(text, audio, vision)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        all_labels.extend(y.detach().cpu().numpy())
        all_logits.extend(pred.argmax(dim=1).detach().cpu().numpy())

        t_bar.set_description(f"Epoch {epoch} training | Batch {i}/{n_batch} | Loss {loss.item():.4f}")
        epoch_loss += (loss.item() * args.batch_size)
    epoch_loss = epoch_loss / n_batch
    
    all_labels , all_logits = np.array(all_labels), np.array(all_logits)
    accuracy = accuracy_score(all_labels, all_logits)
    recall = recall_score(all_labels, all_logits, average=None)
    f1 = f1_score(all_labels, all_logits, average=None)
    precision = precision_score(all_labels, all_logits, average=None)
    print(f"accuracy : {accuracy} , recall : {recall} , precision : {precision} , f1 : {f1}")
    
    # val
    model.eval()
    n_batch = len(val_loader)
    val_loss = 0.0
    all_labels, all_logits = [], []
    for i, (text, audio, vision, y) in enumerate(val_loader):
        if audio is None:
            continue
        text = text["input_ids"].to("cuda")
        audio = audio.to("cuda")
        vision = vision["pixel_values"].to("cuda")
        y = y.to("cuda")
        with torch.no_grad():
            pred = model(text, audio, vision)
            loss = criterion(pred, y)
            
            all_labels.extend(y.detach().cpu().numpy())
            all_logits.extend(pred.argmax(dim=1).detach().cpu().numpy())
            val_loss += (loss.item() * args.batch_size)
    val_loss = val_loss / n_batch
    
    all_labels , all_logits = np.array(all_labels), np.array(all_logits)
    accuracy = accuracy_score(all_labels, all_logits)
    recall = recall_score(all_labels, all_logits, average=None)
    f1 = f1_score(all_labels, all_logits, average=None)
    precision = precision_score(all_labels, all_logits, average=None)
    print(f"accuracy : {accuracy} , recall : {recall} , precision : {precision} , f1 : {f1}")
    
    time_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()).replace(" ", "_")
    run_dir = f"./runs/{time_str}_{args['modal']}"
    os.makedirs(run_dir, exist_ok=True)
    save_path = os.path.join(run_dir, f"epoch_{epoch}.pt")
    torch.save(model.state_dict(), save_path)
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**2:.2f} MB")


In [2]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoProcessor, AutoImageProcessor

# Initialize processors (for compatibility)
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
audio_processor = AutoProcessor.from_pretrained("facebook/hubert-large-ls960-ft")
video_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")

class MultiModalDataset(Dataset):
    def __init__(self, data_root, split, modal="all", tokenizer=None, audio_processor=None, video_processor=None):
        assert split in ["train", "val", "test"]
        assert modal in ["audio", "video", "text", "all"]
        self.modal = modal
        
        # Load metadata
        features_dir = os.path.join(data_root, "features", split)
        metadata_path = os.path.join(features_dir, "metadata.csv")
        
        if not os.path.exists(metadata_path):
            raise FileNotFoundError(
                f"Metadata file not found at {metadata_path}. "
                f"Please run feature extraction first."
            )
        
        df = pd.read_csv(metadata_path)
        
        self.data_list = []
        for _, row in df.iterrows():
            feature_path = os.path.join(features_dir, row["feature_file"])
            self.data_list.append({
                "feature_path": feature_path,
                "label": row["label"]
            })

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        data = self.data_list[idx]
        
        # Load precomputed features
        features = torch.load(data["feature_path"] , weights_only=False)
        
        out_dict = {"label": features["label"]}
        
        if self.modal in ["text", "all"]:
            out_dict["text"] = features["text"]
        
        if self.modal in ["audio", "all"]:
            out_dict["audio"] = features["audio"]
        
        if self.modal in ["video", "all"]:
            out_dict["video"] = features["video"]
        
        return out_dict

def collate_fn(batch):
    out_tuple = ()
    
    if "text" in batch[0]:
        input_ids = [torch.tensor(x["text"]["input_ids"], dtype=torch.long) for x in batch]
        attention_mask = [torch.tensor(x["text"]["attention_mask"], dtype=torch.long) for x in batch]
        text = {
            "input_ids": torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0),
            "attention_mask": torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0),
        }
        out_tuple += (text,)
    
    if "audio" in batch[0]:
        audio_seqs = [torch.tensor(x["audio"], dtype=torch.float) for x in batch]
        audio = torch.nn.utils.rnn.pad_sequence(audio_seqs, batch_first=True, padding_value=0.0)
        out_tuple += (audio,)
    
    if "video" in batch[0]:
        video = {
            "pixel_values": torch.stack([x["video"] for x in batch]),
        }
        out_tuple += (video,)
    
    label = torch.tensor([x["label"] for x in batch])
    out_tuple += (label,)
    
    return out_tuple

if __name__ == "__main__":
    data_root = "./dataset"

    test_set = MultiModalDataset(data_root, "test", modal="all", tokenizer=tokenizer, audio_processor=audio_processor, video_processor=video_processor)

    test_loader = DataLoader(test_set, batch_size=2, shuffle=False, collate_fn=collate_fn)

    text, audio, video, label = next(iter(test_loader))

    print(text["input_ids"].shape)
    print(audio.shape)
    print(video["pixel_values"].shape)
    print(label)

torch.Size([2, 38])
torch.Size([2, 188433])
torch.Size([2, 16, 3, 224, 224])
tensor([0, 0])


/tmp/ipykernel_9521/2445440357.py:74: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio_seqs = [torch.tensor(x["audio"], dtype=torch.float) for x in batch]
